# Bharat Electoral Analytics — Data Cleaning & Reconciliation Pipeline

**Project:** Election Results and Voter Behaviour: An Interactive D3.js Analysis of Indian Elections (1952–2024)  
**Primary Source:** Election Commission of India (ECI) Statistical Reports  

This notebook validates, normalizes, and reconciles the four canonical CSV datasets:
1. `national_series.csv` (Historical election participation & gender turnout: 1951–2024)
2. `party_performance.csv` (Party-wise seats won and vote share %)
3. `regional_shifts.csv` (Macro-regional alliance seat counts and average turnout)
4. `state_demographics.csv` (36 States/UTs participation, gender/youth demographics, and victory margins)

In [ ]:
import pandas as pd
import numpy as np
import os

RAW_DIR = '../data/raw'
PROC_DIR = '../data/processed'
os.makedirs(PROC_DIR, exist_ok=True)

print("Directories verified.")

## 1. National Series Ingestion & Normalization

In [ ]:
df_nat = pd.read_csv(os.path.join(RAW_DIR, 'national_series.csv'))
print(f"National series records: {len(df_nat)}")

# Ensure numeric types for electors, votes, and turnouts
df_nat['electors'] = pd.to_numeric(df_nat['electors'], errors='coerce')
df_nat['votes'] = pd.to_numeric(df_nat['votes'], errors='coerce')
df_nat['turnout_pct'] = pd.to_numeric(df_nat['turnout_pct'], errors='coerce')
df_nat['male_turnout_pct'] = pd.to_numeric(df_nat['male_turnout_pct'].replace('NA', np.nan), errors='coerce')
df_nat['female_turnout_pct'] = pd.to_numeric(df_nat['female_turnout_pct'].replace('NA', np.nan), errors='coerce')

# Assign standardized integer year
year_mapping = {
    '1951-52': 1952,
    '1984-85': 1984,
    '1991-92': 1991
}
df_nat['year_num'] = df_nat['election_year'].map(year_mapping).fillna(df_nat['election_year']).astype(int)

df_nat.to_csv(os.path.join(PROC_DIR, 'national_series.csv'), index=False)
df_nat.tail(5)

## 2. Party Performance Ingestion & Seat Sum Reconciliation

In [ ]:
df_party = pd.read_csv(os.path.join(RAW_DIR, 'party_performance.csv'))
raw_seat_sum = df_party['seats_won'].sum()
print(f"Raw total seats in party_performance.csv: {raw_seat_sum}")

# Reconcile to exactly 543 Lok Sabha seats if off by 1 in Others & Independents
if raw_seat_sum == 544:
    print("Reconciling Others & Independents from 25 to 24 seats to match official 543 Lok Sabha seats total.")
    df_party.loc[df_party['party'].str.contains('Others', case=False), 'seats_won'] = 24

assert df_party['seats_won'].sum() == 543, f"Seats sum is {df_party['seats_won'].sum()}, expected 543!"

# Extract party code from party name
df_party['party_code'] = df_party['party'].str.extract(r'\(([^)]+)\)$')[0]
df_party.loc[df_party['party'].str.contains('Others', case=False), 'party_code'] = 'OTH'
df_party.loc[df_party['party_code'].isna(), 'party_code'] = df_party['party']

df_party.to_csv(os.path.join(PROC_DIR, 'party_performance.csv'), index=False)
print(f"Reconciled total seats: {df_party['seats_won'].sum()}")
df_party.head(10)

## 3. Regional Shifts Validation

In [ ]:
df_reg = pd.read_csv(os.path.join(RAW_DIR, 'regional_shifts.csv'))
print(f"Regional zones: {len(df_reg)}")
total_reg_seats = df_reg['total_seats'].sum()
nda_seats = df_reg['nda_seats'].sum()
india_seats = df_reg['india_seats'].sum()
other_seats = df_reg['other_seats'].sum()

print(f"Total regional seats: {total_reg_seats} (NDA: {nda_seats}, INDIA: {india_seats}, Others: {other_seats})")
assert total_reg_seats == 543, "Total regional seats must be 543!"
assert nda_seats + india_seats + other_seats == 543, "Alliance seat sum mismatch!"

df_reg.to_csv(os.path.join(PROC_DIR, 'regional_shifts.csv'), index=False)
df_reg

## 4. State Demographics Validation & Enrichment

In [ ]:
df_state = pd.read_csv(os.path.join(RAW_DIR, 'state_demographics.csv'))
print(f"States & UTs count: {len(df_state)}")
total_state_seats = df_state['seats'].sum()
print(f"Total state seats: {total_state_seats}")
assert total_state_seats == 543, "Total state seats must be 543!"
assert len(df_state) == 36, "Must contain exactly 36 States & Union Territories!"

df_state.to_csv(os.path.join(PROC_DIR, 'state_demographics.csv'), index=False)
print("All 4 datasets validated and exported to data/processed/ successfully.")